# 01. Из PDF в чанки и поисковый индекс

**Цель:** увидеть, какие данные попадут в поиск и как они получены.
Здесь мы разбираем *Attention Is All You Need*, исследуем границы чанков
и сохраняем их в Qdrant. Генератор и судья пока не нужны.

```text
PDF → Docling + Xiaomi VLM → HybridChunker → чанки
                                          ├→ dense embeddings ─┐
                                          └→ sparse BM25 ──────┴→ Qdrant
```

Затем: [02 — поиск и inference](02_hybrid_rag_inference.ipynb) →
[03 — Ragas](03_hybrid_rag_ragas.ipynb). [Подключение сервисов](SETUP.md).
Python считаем знакомым; устройство RAG разбираем по шагам.
**Run All обращается к embeddings и VLM, может скачивать модели Docling/BM25 и записывает индекс.**
Сохранённых реальных выводов в файле нет: таблицы появятся после выполнения.

## 1. Настройки и необходимые сервисы

Нужны Qdrant, сервер embeddings и Xiaomi VLM. Токенизатор берём у той же модели,
чтобы лимит чанка измерялся в её токенах. По умолчанию `vision=True`: VLM описывает картинки до чанкирования.
Имена нового учебного индекса отделены от прежних `attention_text_v1`.
После изменения корпуса, модели или чанкирования выберите новые `INDEX_NAME` и `COLLECTION`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "helpers" / "artifacts.py").exists():
    ROOT = ROOT / "hybrid_rag_notebooks"
if not (ROOT / "helpers" / "artifacts.py").exists():
    raise RuntimeError("Запустите Jupyter из hybrid_rag_notebooks или её родительской папки")
sys.path.insert(0, str(ROOT))
from helpers.settings import prepare_environment  # noqa: E402

prepare_environment(ROOT)

In [ ]:
import json

import pandas as pd
from helpers.artifacts import (
    append_jsonl,
    ensure_manifest,
    fingerprint,
    latest_records,
    write_json,
)
from helpers.model_clients import ModelAPI
from helpers.parsing import discover_documents, document_inventory, make_tokenizer
from helpers.retrieval import point_id, prepare_collection, sparse_vector
from helpers.settings import model_server_commands, rag_settings
from tqdm.auto import tqdm

CFG = rag_settings()
CFG.update(max_tokens=512, vision=True, translate=False)
INDEX_NAME = "attention_multimodal_v1"
COLLECTION = "notebook_attention_multimodal_v1"
INDEX_DIR = ROOT / "outputs" / "indexes" / INDEX_NAME
DOCUMENTS_DIR = ROOT / "data" / "documents"

# Печать команды не запускает сервер. Запустите его в отдельном терминале.
commands = model_server_commands({**CFG, "rerank": False})
for role, command in commands.items():
    print(role, command, sep="\n")

In [ ]:
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from qdrant_client import models as qm


def close_index_clients():
    for client_name in ("embedder", "vision", "qdrant"):
        client = globals().pop(client_name, None)
        if client is not None:
            client.close()


# Повтор ячейки закрывает прежние подключения перед созданием новых.
close_index_clients()
embedder = vision = qdrant = None
try:
    embedder = ModelAPI(CFG["embed_url"], CFG["embed_model"], "EMBED_API_KEY")
    available = [m["id"] for m in embedder.models()["data"]]
    if embedder.model not in available:
        raise ValueError(f"Модель embeddings не найдена: {available}")
    if CFG["vision"]:
        vision = ModelAPI(CFG["vision_url"], CFG["vision_model"], "VISION_API_KEY")
        available = [m["id"] for m in vision.models()["data"]]
        if vision.model not in available:
            raise ValueError(f"Модель VLM не найдена: {available}")
    dimension = len(embedder.embed(["dimension probe"])[0])
    tokenizer = make_tokenizer(embedder, CFG["max_tokens"], CFG["tokenizer_hf"])
    qdrant = QdrantClient(url=CFG["qdrant_url"], timeout=120)
    qdrant.get_collections()
    print("Размерность dense:", dimension, "лимит токенов:", tokenizer.get_max_tokens())
    print("Загрузка Qdrant/bm25; в первый раз возможно скачивание…", flush=True)
    bm25 = SparseTextEmbedding(CFG["sparse_model"])
except Exception:
    close_index_clients()
    raise

`ModelAPI` оставляет в helpers HTTP, повторы и нормализацию embeddings.
`make_tokenizer` создаёт адаптер: по умолчанию он вызывает **корневой `/tokenize`**
llama.cpp, а embeddings используют `/v1/embeddings`. Если задан `tokenizer_hf`,
используется соответствующий Hugging Face tokenizer.

## 2. PDF → структурированный документ Docling

PDF хранит расположение символов на страницах. Docling восстанавливает текстовые
блоки, заголовки, таблицы и связь с исходными страницами. Чанкирование идёт **после** этого.
OCR распознаёт текст на сканах. Xiaomi VLM описывает рисунки, графики и схемы; описания добавляются в DoclingDocument до сохранения и чанкирования. Ошибка VLM не позволяет сохранить неполный документ.

In [ ]:
paths = discover_documents(DOCUMENTS_DIR)
inventory = document_inventory(paths, DOCUMENTS_DIR)
index_keys = (
    "embed_url",
    "embed_model",
    "doc_template",
    "query_template",
    "sparse_model",
    "max_tokens",
    "device",
    "ocr",
    "vision",
    "vision_url",
    "vision_model",
    "vision_min_area",
    "vision_skip",
    "tokenizer_hf",
)
INDEX_CONFIG = {key: CFG[key] for key in index_keys}
INDEX_CONFIG.update(
    dimension=dimension,
    corpus=inventory,
    collection=COLLECTION,
    qdrant_url=CFG["qdrant_url"],
    implementation=3,
)
ensure_manifest(INDEX_DIR / "index.json", INDEX_CONFIG)
print(pd.DataFrame(inventory).to_string(index=False))

In [ ]:
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import (
    ConvertPipelineOptions,
    OcrAutoOptions,
    PdfPipelineOptions,
)
from docling.document_converter import (
    DocumentConverter,
    ImageFormatOption,
    PdfFormatOption,
    PowerpointFormatOption,
    WordFormatOption,
)
from docling_core.types.doc import DoclingDocument
from helpers.parsing import add_image_descriptions

options = PdfPipelineOptions(
    do_ocr=CFG["ocr"],
    ocr_options=OcrAutoOptions(lang=["iso:ru", "iso:en"]),
    generate_picture_images=CFG.get("vision", False),
    images_scale=2.0,
    do_picture_classification=CFG.get("vision", False) and bool(CFG["vision_skip"]),
    accelerator_options=AcceleratorOptions(device=AcceleratorDevice(CFG["device"])),
)
formats = {
    InputFormat.PDF: PdfFormatOption(pipeline_options=options),
    InputFormat.IMAGE: ImageFormatOption(pipeline_options=options),
}
if CFG.get("vision"):
    if vision is None:
        raise RuntimeError("Сначала выполните подключение к VLM")
    office = ConvertPipelineOptions(do_picture_classification=bool(CFG["vision_skip"]))
    formats[InputFormat.DOCX] = WordFormatOption(pipeline_options=office)
    formats[InputFormat.PPTX] = PowerpointFormatOption(pipeline_options=office)
converter = DocumentConverter(format_options=formats)
documents = []
print(f"Файлов: {len(paths)}. Готовые Docling JSON читаем с диска.", flush=True)
for path, entry in tqdm(
    list(zip(paths, inventory, strict=True)), desc="Docling + VLM", unit="файл"
):
    doc_id = fingerprint(entry)[:16]
    cache = INDEX_DIR / "docling" / f"{doc_id}.json"
    if cache.exists():
        doc = DoclingDocument.model_validate_json(cache.read_text(encoding="utf-8"))
    else:
        print(f"Разбор {entry['path']}; журнал Docling показывает работу внутри файла…", flush=True)
        conversion = converter.convert(path)
        if conversion.status != ConversionStatus.SUCCESS:
            raise RuntimeError(f"{path}: Docling {conversion.status.value}; {conversion.errors}")
        doc = conversion.document
        if CFG.get("vision"):
            doc = add_image_descriptions(
                doc, path, vision, min_area=CFG["vision_min_area"], skip=CFG["vision_skip"]
            )
        # Сохраняем только после успешных описаний всех выбранных картинок.
        write_json(cache, doc.model_dump(mode="json"))
    documents.append((entry, doc_id, doc))

entry, doc_id, doc = documents[0]
print(doc.export_to_markdown()[:4000])
print("Таблиц:", len(doc.tables), "изображений:", len(doc.pictures))
print("Описаний VLM:", sum(item.text.startswith("Image description:") for item in doc.texts))
if doc.tables:
    print(doc.tables[0].export_to_dataframe(doc=doc).to_string(index=False))

### Сверим извлечение с исходной страницей

Для статьи откройте страницу 5: раздел 3.2.2 содержит число голов и размерности.
Изображение ниже — реальная страница PDF; рядом выводим текст Docling с той же страницы.
После смены корпуса выберите нужную страницу. Таблицу 3 статьи полезно проверить на странице 9.

In [ ]:
PAGE = 5  # Номер PDF-страницы с единицы.
source_path = DOCUMENTS_DIR / entry["path"]
if source_path.suffix.lower() == ".pdf":
    import pypdfium2 as pdfium
    from IPython.display import display

    with pdfium.PdfDocument(source_path) as pdf:
        if not 1 <= PAGE <= len(pdf):
            raise ValueError(f"PAGE должен быть от 1 до {len(pdf)}")
        page = pdf[PAGE - 1]
        bitmap = page.render(scale=1.2)
        image = bitmap.to_pil().copy()
        display(image)
        bitmap.close()
        page.close()
    print(doc.export_to_markdown(page_no=PAGE))
else:
    print("Это не PDF; сравните Markdown выше с исходным файлом:", source_path)

## 3. Структурированный документ → чанки

**Инструмент — `docling_core…HybridChunker`.** Сначала он учитывает структуру
документа, затем токеновый лимит; `merge_peers=True` разрешает объединять подходящие
соседние части. Это не фиксированное окно с заданным overlap.
Название чанкера не связано с объединением Dense и BM25 в следующем ноутбуке.

`raw.text` — содержимое чанка; `contextualize(raw)` добавляет доступные заголовки
и другие метаданные. Лимит проверяется токенизатором, а не числом символов.
В нашем совместимом payload храним текст и заголовки раздельно: embeddings/BM25
индексируют `text`; reranker и сборка контекста используют также заголовки.

In [ ]:
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker


def split_document(doc, path, doc_id, tokenizer):
    chunker = HybridChunker(tokenizer=tokenizer, merge_peers=True)
    records = []
    for raw in chunker.chunk(doc):
        if not raw.text.strip():
            continue
        pages = sorted({p.page_no for item in raw.meta.doc_items for p in item.prov})
        records.append(
            {
                "chunk_id": f"{doc_id}:{len(records):05d}",
                "doc_id": doc_id,
                "idx": len(records),
                "text": raw.text.strip(),
                "headings": list(raw.meta.headings or []),
                "page": pages[0] if pages else None,
                "path": path,
                "n_tokens": tokenizer.count_tokens(chunker.contextualize(raw)),
            }
        )
    if not records:
        raise ValueError(f"{path}: документ не дал текстовых чанков")
    return records

In [ ]:
chunks = []
for entry, doc_id, doc in tqdm(documents, desc="Чанкирование", unit="документ"):
    cache = INDEX_DIR / "parsed" / f"{doc_id}.json"
    if cache.exists():
        records = json.loads(cache.read_text(encoding="utf-8"))
    else:
        records = split_document(doc, entry["path"], doc_id, tokenizer)
        write_json(cache, records)
    chunks.extend(records)
print("Всего чанков:", len(chunks))
print(
    pd.DataFrame(chunks)[["chunk_id", "page", "headings", "n_tokens"]]
    .head(12)
    .to_string(index=False)
)

In [ ]:
CHUNK_NUMBER = 0  # Измените и повторите только эту ячейку.
preview_chunk = chunks[CHUNK_NUMBER]
print(json.dumps(preview_chunk, ensure_ascii=False, indent=2))

# Отдельно смотрим объект чанкера и текст с добавленными метаданными.
first_entry, first_id, first_doc = documents[0]
chunker = HybridChunker(tokenizer=tokenizer, merge_peers=True)
raw_chunks = list(chunker.chunk(first_doc))
print("Исходный документ, начало:\n", first_doc.export_to_markdown()[:1200])
print("Первый raw.text:\n", raw_chunks[0].text)
print("Первый contextualize(raw):\n", chunker.contextualize(raw_chunks[0]))

### Эксперимент: 256 против 512 токенов

Повторяем только чанкирование первого документа, без повторного OCR и индексации.
Проверьте не только число чанков: остались ли строки таблицы рядом с названием
варианта модели? Длинный чанк даёт больше контекста, короткий позволяет точнее выбрать фрагмент.
Большие неделимые элементы могут выходить за лимит — ниже явно считаем такие случаи.

In [ ]:
comparison = []
for limit in tqdm([256, 512], desc="Сравнение лимитов", unit="вариант"):
    trial_tokenizer = make_tokenizer(embedder, limit, CFG["tokenizer_hf"])
    trial = split_document(first_doc, first_entry["path"], first_id, trial_tokenizer)
    lengths = pd.Series([c["n_tokens"] for c in trial])
    comparison.append(
        {
            "limit": limit,
            "chunks": len(trial),
            "min": lengths.min(),
            "median": lengths.median(),
            "max": lengths.max(),
            "over_limit": int((lengths > limit).sum()),
        }
    )
    print(f"\nЛимит {limit}: распределение длин")
    print(
        pd.cut(lengths, bins=[0, 128, 256, 512, float("inf")]).value_counts(sort=False).to_string()
    )
    table_chunks = [c for c in trial if "Table 3" in c["text"] or "d_model" in c["text"]]
    for c in table_chunks[:2]:
        print(c["chunk_id"], c["n_tokens"], c["text"][:1500], sep="\n")
print(pd.DataFrame(comparison).to_string(index=False))
print("Основной индекс остаётся с лимитом:", CFG["max_tokens"])

## 4. Чанки → dense и sparse → Qdrant

Dense — вектор фиксированной размерности для смыслового поиска.
BM25 — разреженные веса терминов: `indices` обозначают термины во внутреннем
представлении модели, `values` — их веса. Qdrant дополнительно применяет IDF
по коллекции. Ни dense-координаты, ни sparse-индексы не являются читаемым текстом.

Создаём именованные векторные поля `dense` (cosine) и `bm25` (IDF).
Существующая чужая коллекция без привязки к нашему манифесту отвергается.

In [ ]:
existing = qdrant.collection_exists(COLLECTION)
if existing and not (INDEX_DIR / "binding.json").exists():
    raise ValueError(
        "Коллекция уже существует без привязки. Выберите новые INDEX_NAME и COLLECTION"
    )
ensure_manifest(INDEX_DIR / "binding.json", INDEX_CONFIG)
if not existing:
    qdrant.create_collection(
        COLLECTION,
        vectors_config={"dense": qm.VectorParams(size=dimension, distance=qm.Distance.COSINE)},
        sparse_vectors_config={"bm25": qm.SparseVectorParams(modifier=qm.Modifier.IDF)},
    )
    qdrant.create_payload_index(COLLECTION, "chunk_id", field_schema=qm.PayloadSchemaType.KEYWORD)
else:
    # Только проверка схемы существующей коллекции; создание показано выше.
    prepare_collection(qdrant, COLLECTION, dimension)
print("Коллекция готова:", COLLECTION)

In [ ]:
finished = latest_records(INDEX_DIR / "indexed.jsonl")
present = set()
ids = [point_id(c["chunk_id"]) for c in chunks]
for start in tqdm(range(0, len(ids), 256), desc="Проверка точек", unit="пакет"):
    present.update(
        str(p.id)
        for p in qdrant.retrieve(COLLECTION, ids=ids[start : start + 256], with_payload=False)
    )
pending = [
    c
    for c in chunks
    if point_id(c["chunk_id"]) not in present
    or finished.get(c["chunk_id"], {}).get("status") != "ok"
]
print(f"Готово: {len(chunks) - len(pending)}; осталось: {len(pending)}", flush=True)

# Первый пакет — один объект: ниже можно исследовать его вектора и payload.
preview_point = None
batches = (
    [pending[:1]]
    + [pending[i : i + CFG["batch_size"]] for i in range(1, len(pending), CFG["batch_size"])]
    if pending
    else []
)
for batch in tqdm(batches, desc="Embeddings + upsert", unit="пакет"):
    texts = [CFG["doc_template"].format(text=c["text"], title=Path(c["path"]).stem) for c in batch]
    dense = embedder.embed(texts, dim=dimension)
    sparse = [sparse_vector(e) for e in bm25.passage_embed([c["text"] for c in batch])]
    points = [
        qm.PointStruct(id=point_id(c["chunk_id"]), vector={"dense": d, "bm25": s}, payload=c)
        for c, d, s in zip(batch, dense, sparse, strict=True)
    ]
    if preview_point is None:
        preview_point = points[0]
        print("Dense:", len(dense[0]), "координат; начало:", dense[0][:8])
        print("BM25 indices:", sparse[0].indices[:8], "values:", sparse[0].values[:8])
    qdrant.upsert(COLLECTION, points=points, wait=True)
    for c in batch:
        append_jsonl(INDEX_DIR / "indexed.jsonl", {"id": c["chunk_id"], "status": "ok"})
count = qdrant.count(COLLECTION, exact=True).count
if count != len(chunks):
    raise ValueError(f"В коллекции {count} точек, ожидалось {len(chunks)}. Нужна новая коллекция")
print("Индекс готов, точек:", count)

In [ ]:
point = qdrant.retrieve(COLLECTION, ids=[point_id(chunks[0]["chunk_id"])], with_vectors=True)[0]
print("ID:", point.id, "payload:", json.dumps(point.payload, ensure_ascii=False, indent=2))
print("Dense, первые координаты:", point.vector["dense"][:8])
print("BM25, первые индексы:", point.vector["bm25"].indices[:8])
print("Манифест для второго ноутбука:", INDEX_DIR / "index.json")

## Что получилось

На диске лежат Docling JSON, чанки и манифест индекса; в Qdrant — векторы и payload.
Повторный запуск использует сохранённые документы и проверяет наличие точек.
После прерывания повторите ячейку; после перезапуска kernel выполняйте сверху вниз.

**Проверка понимания:** найдите чанк с `Table 3`, сверьте числа и подпись с PDF.
Если ошибка появилась при извлечении, изменение генератора её не исправит.
Переходите к [02 — поиск и inference](02_hybrid_rag_inference.ipynb) с тем же `INDEX_NAME`.
По умолчанию индекс включает текст, OCR и описания картинок от Xiaomi.
Для явного отключения задайте `CFG['vision'] = False` до подключения и выберите другое имя индекса.
В 02 описания уже доступны в поиске; повторный вызов VLM для вопроса не нужен.

In [ ]:
if "close_index_clients" in globals():
    close_index_clients()